# 02. Equivalent Circuit Model (ECM) Parameter Estimation
This notebook covers optimization steps to estimate cycle-resolved parameters for the 2-RC Equivalent Circuit Model.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.append('..')

from src.config import BATTERY_FILES
from src.data_loader import load_cycles, cycle_to_df, is_discharge, compute_capacity_Ah

# Load a discharge cycle for cell B0005
cycles = load_cycles(BATTERY_FILES['B0005'])
discharge_cycles = [c for c in cycles if is_discharge(c)]
df_cycle = cycle_to_df(discharge_cycles[0])

In [ ]:
from src.parameter_estimation import fit_ecm

# Fit simple 5-parameter baseline model
t = df_cycle['Time (s)'].values
Vm = df_cycle['Voltage (V)'].values
I = df_cycle['Current (A)'].values

p_simple = fit_ecm(I, Vm, t)
print("Fitted simple baseline R0 parameter:", p_simple)

In [ ]:
from src.parameter_estimation import fit_cycle_2rc

# Fit physics-informed 7-parameter 2-RC model
Qnom = compute_capacity_Ah(df_cycle)
p_2rc, success = fit_cycle_2rc(I, Vm, t, Qnom)
print("Fitted 2-RC parameters (R0, R1, C1, R2, C2, Voc_slope, Voc_intercept):")
print(p_2rc)
print("Optimization Success:", success)

In [ ]:
from src.ecm_model import simulate_ecm_2rc

# Run simulation with identified parameters
dt = np.diff(t, prepend=t[0])
Vsim = simulate_ecm_2rc(p_2rc, I, dt, Qnom)

# Plot comparison
plt.figure(figsize=(10, 5))
plt.plot(t, Vm, 'k-', label='Measured Voltage')
plt.plot(t, Vsim, 'r--', label='Simulated 2-RC Voltage')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.title('Comparison between Measured and Simulated Voltages')
plt.legend()
plt.grid(True)
plt.show()